# 08. 전체 데이터 기반 평가 — TF-IDF vs BERT

**목표**: validation 2,399건 전체를 쿼리로 사용하여 TF-IDF와 BERT의 성능을 비교하고  
McNemar test로 통계적 유의성을 검정합니다.

---

## 평가 설계

| 구분 | 내용 |
|------|------|
| **인덱스 (검색 대상)** | train set 19,205건 |
| **쿼리 (평가 세트)** | validation set 2,399건 |
| **소프트 매치 기준** | 검색 결과의 department + lifeCycle이 쿼리와 동일하면 Hit |
| **메트릭** | Hit@1, Hit@3, Hit@5, MAP@5, McNemar test |

### 소프트 매치 근거
- `disease` 컬럼의 76%가 '기타'로 분류되어 exact match 불가능
- `department`(5개 진료과) × `lifeCycle`(3단계) = 15개 임상 카테고리
- 같은 진료과·생애주기 답변을 검색하면 임상적으로 유의미한 결과

In [11]:
import sys
import warnings
import numpy as np
import pandas as pd
from pathlib import Path
from scipy.sparse import issparse
from scipy.stats import chi2

warnings.filterwarnings('ignore')

# 프로젝트 루트를 sys.path에 추가
ROOT = Path('..').resolve()
sys.path.insert(0, str(ROOT))

DATA = ROOT / 'data' / 'processed'
SPLITS = ROOT / 'data' / 'splits'

print('설정 완료')
print('ROOT:', ROOT)

설정 완료
ROOT: /Users/brandi/Desktop/project/pet-health-ai


## 1. 데이터 로드 및 분리

In [2]:
corpus = pd.read_csv(DATA / 'corpus_preprocessed.csv')

train_df = corpus[corpus['split'] == 'train'].reset_index(drop=True)
val_df   = corpus[corpus['split'] == 'validation'].reset_index(drop=True)

print(f'Train: {len(train_df):,}건')
print(f'Validation (쿼리): {len(val_df):,}건')
print()
print('Validation lifeCycle:', val_df['lifeCycle'].value_counts().to_dict())
print('Validation department:', val_df['department'].value_counts().to_dict())

Train: 19,205건
Validation (쿼리): 2,399건

Validation lifeCycle: {'노령견': 809, '성견': 808, '자견': 782}
Validation department: {'내과': 1203, '외과': 651, '피부과': 301, '치과': 129, '안과': 115}


## 2. TF-IDF 인덱스 구축

In [4]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# train set으로 TF-IDF 학습
train_tokens = train_df['input_tokens'].fillna('').tolist()
val_tokens   = val_df['input_tokens'].fillna('').tolist()

tfidf = TfidfVectorizer()
train_tfidf_matrix = tfidf.fit_transform(train_tokens)

print(f'TF-IDF 행렬 크기: {train_tfidf_matrix.shape}')
print(f'어휘 크기: {len(tfidf.vocabulary_):,}')

TF-IDF 행렬 크기: (19205, 10232)
어휘 크기: 10,232


## 3. BERT 임베딩 로드

In [5]:
# train 임베딩 (이미 계산된 것)
train_bert_emb = np.load(DATA / 'embeddings' / 'db_embeddings.npy')
print(f'Train BERT 임베딩: {train_bert_emb.shape}')

# validation 쿼리 임베딩 계산
# ※ conda env pet-health 에서 실행해야 sentence-transformers 사용 가능
val_emb_path = DATA / 'embeddings' / 'val_embeddings.npy'

if val_emb_path.exists():
    val_bert_emb = np.load(val_emb_path)
    print(f'Validation BERT 임베딩 로드: {val_bert_emb.shape}')
else:
    print('Validation 임베딩 미존재 → 계산 시작...')
    from sentence_transformers import SentenceTransformer
    model = SentenceTransformer('jhgan/ko-sroberta-multitask')
    val_texts = val_df['input_normalized'].fillna('').tolist()
    val_bert_emb = model.encode(val_texts, batch_size=64, show_progress_bar=True, normalize_embeddings=True)
    np.save(val_emb_path, val_bert_emb)
    print(f'Validation 임베딩 저장 완료: {val_bert_emb.shape}')

Train BERT 임베딩: (19205, 768)
Validation 임베딩 미존재 → 계산 시작...


Batches:   0%|          | 0/38 [00:00<?, ?it/s]

Validation 임베딩 저장 완료: (2399, 768)


## 4. 소프트 매치 지표 계산 함수

In [12]:
def soft_hit(retrieved_indices: list[int], query_dept: str, query_lc: str, train_df: pd.DataFrame) -> list[bool]:
    """검색된 인덱스별로 소프트 매치(department + lifeCycle 일치) 여부 반환."""
    return [
        train_df.iloc[i]['department'] == query_dept and
        train_df.iloc[i]['lifeCycle']  == query_lc
        for i in retrieved_indices
    ]


def compute_metrics(hits_list: list[list[bool]], K: int = 5) -> dict:
    """Hit@1/3/5, MAP@5 계산. hits_list: 쿼리별 [True/False, ...] 리스트 (길이 K)."""
    hit1, hit3, hit5, ap_sum = 0, 0, 0, 0.0
    n = len(hits_list)

    for hits in hits_list:
        if any(hits[:1]): hit1 += 1
        if any(hits[:3]): hit3 += 1
        if any(hits[:5]): hit5 += 1

        # Average Precision
        prec_sum, n_rel = 0.0, 0
        for k, h in enumerate(hits[:K], 1):
            if h:
                n_rel += 1
                prec_sum += n_rel / k
        ap_sum += prec_sum / max(n_rel, 1)

    return {
        'Hit@1': hit1 / n,
        'Hit@3': hit3 / n,
        'Hit@5': hit5 / n,
        'MAP@5': ap_sum / n,
    }

print('함수 정의 완료')

함수 정의 완료


## 5. TF-IDF 전체 평가 (2,399건)

In [7]:
TOP_K = 5

# validation 쿼리를 TF-IDF 벡터화 (train 어휘 기준)
val_tfidf_matrix = tfidf.transform(val_tokens)

tfidf_hits_list = []
tfidf_hit1_binary = []  # McNemar용 이진 벡터

# 배치 처리 (메모리 효율)
BATCH = 200
for start in range(0, len(val_df), BATCH):
    end = min(start + BATCH, len(val_df))
    batch_q = val_tfidf_matrix[start:end]
    sims = cosine_similarity(batch_q, train_tfidf_matrix)  # (batch, 19205)
    top_indices = np.argsort(-sims, axis=1)[:, :TOP_K]

    for i, row_idx in enumerate(range(start, end)):
        hits = soft_hit(
            top_indices[i].tolist(),
            val_df.iloc[row_idx]['department'],
            val_df.iloc[row_idx]['lifeCycle'],
            train_df
        )
        tfidf_hits_list.append(hits)
        tfidf_hit1_binary.append(1 if hits[0] else 0)

tfidf_metrics = compute_metrics(tfidf_hits_list)
print('=== TF-IDF 전체 평가 결과 ===')
for k, v in tfidf_metrics.items():
    print(f'  {k}: {v:.4f} ({v:.1%})')

=== TF-IDF 전체 평가 결과 ===
  Hit@1: 0.2380 (23.8%)
  Hit@3: 0.5231 (52.3%)
  Hit@5: 0.6782 (67.8%)
  MAP@5: 0.3785 (37.9%)


## 6. BERT 전체 평가 (2,399건)

In [8]:
# L2 정규화 (코사인 유사도 = 내적)
def normalize(v):
    norm = np.linalg.norm(v, axis=1, keepdims=True)
    return v / np.where(norm == 0, 1, norm)

train_emb_norm = normalize(train_bert_emb)
val_emb_norm   = normalize(val_bert_emb)

bert_hits_list = []
bert_hit1_binary = []  # McNemar용 이진 벡터

for start in range(0, len(val_df), BATCH):
    end = min(start + BATCH, len(val_df))
    batch_q = val_emb_norm[start:end]          # (batch, 768)
    sims = batch_q @ train_emb_norm.T          # (batch, 19205)
    top_indices = np.argsort(-sims, axis=1)[:, :TOP_K]

    for i, row_idx in enumerate(range(start, end)):
        hits = soft_hit(
            top_indices[i].tolist(),
            val_df.iloc[row_idx]['department'],
            val_df.iloc[row_idx]['lifeCycle'],
            train_df
        )
        bert_hits_list.append(hits)
        bert_hit1_binary.append(1 if hits[0] else 0)

bert_metrics = compute_metrics(bert_hits_list)
print('=== BERT 전체 평가 결과 ===')
for k, v in bert_metrics.items():
    print(f'  {k}: {v:.4f} ({v:.1%})')

=== BERT 전체 평가 결과 ===
  Hit@1: 0.2468 (24.7%)
  Hit@3: 0.5215 (52.1%)
  Hit@5: 0.6744 (67.4%)
  MAP@5: 0.3829 (38.3%)


## 7. 결과 비교표

In [9]:
result_df = pd.DataFrame({
    'TF-IDF': tfidf_metrics,
    'BERT':   bert_metrics,
}).T

result_df['Hit@1 차이'] = result_df['Hit@1'] - result_df['Hit@1'].iloc[0]
print('=== 전체 평가 비교 (n=2,399) ===')
print(result_df.round(4))

print()
print('--- 50건 기존 평가 참고 ---')
print('TF-IDF Hit@1: 18.0%  BERT Hit@1: 24.0%  (수동 GT 50건)')

=== 전체 평가 비교 (n=2,399) ===
         Hit@1   Hit@3   Hit@5   MAP@5  Hit@1 차이
TF-IDF  0.2380  0.5231  0.6782  0.3785    0.0000
BERT    0.2468  0.5215  0.6744  0.3829    0.0088

--- 50건 기존 평가 참고 ---
TF-IDF Hit@1: 18.0%  BERT Hit@1: 24.0%  (수동 GT 50건)


## 8. McNemar Test — 통계적 유의성 검정 (Q1 답변)

In [10]:
# 2×2 분할표 구성
tfidf_arr = np.array(tfidf_hit1_binary)  # n=2399
bert_arr  = np.array(bert_hit1_binary)

both_hit  = int(np.sum((tfidf_arr == 1) & (bert_arr == 1)))  # 둘 다 맞춤
tfidf_only = int(np.sum((tfidf_arr == 1) & (bert_arr == 0)))  # TF-IDF만 맞춤 (b)
bert_only  = int(np.sum((tfidf_arr == 0) & (bert_arr == 1)))  # BERT만 맞춤 (c)
both_miss  = int(np.sum((tfidf_arr == 0) & (bert_arr == 0)))  # 둘 다 틀림

print('=== 2×2 분할표 ===')
print(f'           BERT 맞춤   BERT 틀림')
print(f'TF-IDF 맞춤   {both_hit:4d}       {tfidf_only:4d}')
print(f'TF-IDF 틀림   {bert_only:4d}       {both_miss:4d}')
print()

b, c = tfidf_only, bert_only

# McNemar 검정 (연속성 교정 포함, n<25이면 exact 권장이지만 여기서는 n이 충분히 큼)
if b + c == 0:
    print('경고: b + c = 0, 두 모델 완전 동일')
else:
    # 연속성 교정 (Yates)
    chi2_stat = (abs(b - c) - 1) ** 2 / (b + c)
    from scipy.stats import chi2 as chi2_dist
    p_value = chi2_dist.sf(chi2_stat, df=1)

    print('=== McNemar Test 결과 ===')
    print(f'b (TF-IDF only): {b}')
    print(f'c (BERT only):   {c}')
    print(f'χ² (연속성 교정): {chi2_stat:.4f}')
    print(f'p-value: {p_value:.4f}')
    print()
    if p_value < 0.05:
        print('✅ p < 0.05: BERT와 TF-IDF의 Hit@1 차이가 통계적으로 유의미합니다.')
    elif p_value < 0.10:
        print('⚠️  p < 0.10: 경계선 유의미 (n 증가 또는 추가 검증 권장)')
    else:
        print('❌ p ≥ 0.05: 통계적으로 유의미한 차이 없음 → 차이가 우연일 수 있음')

=== 2×2 분할표 ===
           BERT 맞춤   BERT 틀림
TF-IDF 맞춤    213        358
TF-IDF 틀림    379       1449

=== McNemar Test 결과 ===
b (TF-IDF only): 358
c (BERT only):   379
χ² (연속성 교정): 0.5427
p-value: 0.4613

❌ p ≥ 0.05: 통계적으로 유의미한 차이 없음 → 차이가 우연일 수 있음


## 9. Paired Bootstrap 신뢰구간 (McNemar 보완)

In [13]:
np.random.seed(42)
N_BOOTSTRAP = 2000
n = len(tfidf_arr)

diffs = []
for _ in range(N_BOOTSTRAP):
    idx = np.random.randint(0, n, size=n)
    d = bert_arr[idx].mean() - tfidf_arr[idx].mean()
    diffs.append(d)

diffs = np.array(diffs)
ci_lo, ci_hi = np.percentile(diffs, [2.5, 97.5])

observed_diff = bert_arr.mean() - tfidf_arr.mean()

print('=== Paired Bootstrap 결과 (2,000회) ===')
print(f'BERT - TF-IDF Hit@1 실제 차이: {observed_diff:+.4f} ({observed_diff:+.1%})')
print(f'95% 신뢰구간: [{ci_lo:.4f}, {ci_hi:.4f}]')
print()
if ci_lo > 0:
    print('✅ 신뢰구간 하한 > 0: BERT가 TF-IDF보다 유의미하게 우수')
elif ci_hi < 0:
    print('TF-IDF가 BERT보다 유의미하게 우수')
else:
    print('신뢰구간이 0을 포함 → 유의미한 차이 없음')

=== Paired Bootstrap 결과 (2,000회) ===
BERT - TF-IDF Hit@1 실제 차이: +0.0088 (+0.9%)
95% 신뢰구간: [-0.0125, 0.0308]

신뢰구간이 0을 포함 → 유의미한 차이 없음


## 10. 생애주기별 세부 분석

In [14]:
val_df = val_df.copy()
val_df['tfidf_hit1'] = tfidf_hit1_binary
val_df['bert_hit1']  = bert_hit1_binary

print('=== 생애주기별 Hit@1 ===')
for lc in ['자견', '성견', '노령견']:
    sub = val_df[val_df['lifeCycle'] == lc]
    n_lc = len(sub)
    t = sub['tfidf_hit1'].mean()
    b = sub['bert_hit1'].mean()
    winner = 'BERT ↑' if b > t else ('TF-IDF ↑' if t > b else '동률')
    print(f'  {lc} (n={n_lc}): TF-IDF {t:.1%} vs BERT {b:.1%} → {winner} ({b-t:+.1%})')

print()
print('=== 진료과별 Hit@1 ===')
for dept in val_df['department'].unique():
    sub = val_df[val_df['department'] == dept]
    n_dept = len(sub)
    t = sub['tfidf_hit1'].mean()
    b = sub['bert_hit1'].mean()
    winner = 'BERT ↑' if b > t else ('TF-IDF ↑' if t > b else '동률')
    print(f'  {dept} (n={n_dept}): TF-IDF {t:.1%} vs BERT {b:.1%} → {winner} ({b-t:+.1%})')

=== 생애주기별 Hit@1 ===
  자견 (n=782): TF-IDF 25.4% vs BERT 23.7% → TF-IDF ↑ (-1.8%)
  성견 (n=808): TF-IDF 24.6% vs BERT 26.4% → BERT ↑ (+1.7%)
  노령견 (n=809): TF-IDF 21.4% vs BERT 24.0% → BERT ↑ (+2.6%)

=== 진료과별 Hit@1 ===
  외과 (n=651): TF-IDF 23.0% vs BERT 20.7% → TF-IDF ↑ (-2.3%)
  내과 (n=1203): TF-IDF 24.8% vs BERT 26.0% → BERT ↑ (+1.2%)
  안과 (n=115): TF-IDF 15.7% vs BERT 30.4% → BERT ↑ (+14.8%)
  피부과 (n=301): TF-IDF 24.9% vs BERT 26.9% → BERT ↑ (+2.0%)
  치과 (n=129): TF-IDF 23.3% vs BERT 21.7% → TF-IDF ↑ (-1.6%)


## 11. 결과 저장

In [15]:
# 전체 평가 결과 저장
result_df.to_csv(DATA / 'full_evaluation_summary.csv')

# 쿼리별 hit 결과 저장
val_df[['query_col' if 'query_col' in val_df else col for col in val_df.columns if col in ['lifeCycle','department','disease','tfidf_hit1','bert_hit1']]].to_csv(
    DATA / 'full_matching_results.csv', index=False
)

print('저장 완료')
print('  data/processed/full_evaluation_summary.csv')
print('  data/processed/full_matching_results.csv')

저장 완료
  data/processed/full_evaluation_summary.csv
  data/processed/full_matching_results.csv


## 정리 — 교수님 Q1 답변

| 구분 | 내용 |
|------|------|
| **검정 방법** | McNemar test (연속성 교정) + Paired Bootstrap 2,000회 |
| **표본 크기** | n = 2,399 (validation 전체, 기존 50건 대비 48배) |
| **소프트 매치** | department + lifeCycle (임상적 유의미성 기준) |
| **결론** | p-value 및 신뢰구간 해석 → 위 출력 결과 참조 |